# 03 — Metamodel Joint Inference

Build the joint metamodel coupling all 4 surrogates and run Bayesian inference.

**Prerequisites**: Trained surrogates from notebook 02.

> **This notebook needs the `bayesian-metamodeling` framework.**
> It drives the `bayesmm` CLI, unlike the kinetic-segregation series under
> `notebooks/models/kinetic_segregation/`, which needs only numpy and the compiled
> model. If `bayesmm` is missing, install the framework
> (`pip install -e .` from the parent repo, or `pip install bayesian-metamodeling`)
> and restart the kernel.
>
> New here? Start at [`Tutorial_0_Start_Here.ipynb`](Tutorial_0_Start_Here.ipynb).

## Learning aims
- **Primary**: build the metamodel IR from coupled surrogates and sample the joint
  posterior.
- **Secondary scientific**: explain what a coupling asserts, and what "joint" buys you
  over four separate posteriors.

## What coupling means

Each partial model has its own posterior. Independently, their joint distribution is
just a product — nothing relates them. A **coupling** states that two variables in
different models are the same physical quantity, or are related by a known transform.

`specs/metamodel.tcr_signaling.json` couples `depletion_width_nm` with a
`gaussian_link` of sigma 10 nm: *these should agree, to within 10 nm*. That constraint
propagates — data that sharpens one model's posterior now sharpens its neighbours too.

**Requires `pymc`.** Sampling is the slow step; the draw counts here are deliberately
modest for interactive use, and production values are noted inline.

In [ ]:
import json
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


def find_repo_root(start=None):
    """Walk upward until we find this repo, instead of assuming a fixed depth.

    The previous version walked up a fixed number of levels from the working
    directory, which silently assumed a submodule checkout and broke in a
    standalone clone or from any other directory. Searching for a landmark is
    robust to both.
    """
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "models" / "kinetic_segregation" / "CMakeLists.txt").is_file():
            return cand
    raise RuntimeError(f"could not locate the tcr_signaling repo above {here}")


ROOT = find_repo_root()
SPECS = ROOT / "specs"
# The metamodel spec every cell below drives. Its assignment was missing, so
# the first `meta build` cell raised NameError — unnoticed because no CI job
# executed these notebooks until `Submodule notebooks CI` was added.
META_SPEC = SPECS / "metamodel.tcr_signaling.json"
print(f"repo root: {ROOT}")

# These notebooks drive the bayesian-metamodeling CLI. Report clearly if absent.
HAVE_BAYESMM = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "--version"], capture_output=True, text=True
).returncode == 0
print("bayesmm:", "available" if HAVE_BAYESMM else "NOT INSTALLED -- see the banner above")


## Build metamodel IR

In [ ]:
r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "meta", "build", str(META_SPEC)],
                   cwd=str(ROOT), capture_output=True, text=True)
print("Return code:", r.returncode)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr)

## Sample from the joint posterior

In [ ]:
r = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "meta", "sample", str(META_SPEC), "--draws", "2000", "--tune", "1000"],
    cwd=str(ROOT), capture_output=True, text=True
)
print("Return code:", r.returncode)
print(r.stdout[:500])
if r.returncode != 0:
    print("STDERR:", r.stderr[:500])

## Inspect posterior samples

In [ ]:
r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "meta", "list"], cwd=str(ROOT),
                   capture_output=True, text=True)
print(r.stdout)

## Final check

In [ ]:
# Self-check: the metamodel really coupled all four partial models and produced a
# posterior — not `assert ROOT.is_dir()`, which was the previous assertion and is
# true in a repo where nothing ever ran.
import json as _json
import numpy as _np
from pathlib import Path as _P

_irs = sorted((ROOT / "tmp/metamodel_ir").glob("*/ir.json"), key=lambda q: q.stat().st_mtime)
assert _irs, "no metamodel IR was built — Step 1's `meta build` did not run"
_ir = _json.loads(_irs[-1].read_text())

_kinds = {}
for _f in _ir["factors"]:
    _kinds[_f["kind"]] = _kinds.get(_f["kind"], 0) + 1
print(f"  IR: {len(_ir['variables'])} variables, {len(_ir['factors'])} factors {_kinds}")

# One surrogate likelihood per partial model, and at least one coupling — a
# metamodel with no coupling factor is four independent models in a trenchcoat.
assert _kinds.get("surrogate_likelihood") == 4, (
    f"expected 4 surrogate likelihoods (one per partial model), got "
    f"{_kinds.get('surrogate_likelihood')} — did all four publish in notebook 02?"
)
assert _kinds.get("coupling", 0) >= 1, "no coupling factors: the models are not constrained by each other"

_samples = sorted((ROOT / "tmp/metamodel_samples").glob("*/samples_dataset.json"),
                  key=lambda q: q.stat().st_mtime)
assert _samples, "no posterior samples — Step 2's `meta sample` did not run"
_ds = _json.loads(_samples[-1].read_text())
_vars = _ds["variables"]

# The variables the chain actually threads through: topography -> lck -> phosphorylation.
for _key in ("contact_fraction", "mean_lck_activity", "ptcr_fraction"):
    assert _key in _vars, f"'{_key}' absent from the posterior; coupled variables: {sorted(_vars)[:8]}"
    _draws = _np.asarray(_vars[_key], dtype=float).ravel()
    assert _draws.size > 100, f"{_key}: only {_draws.size} draws"
    assert _np.all(_np.isfinite(_draws)), f"{_key}: posterior contains NaN/inf"
    # A collapsed sampler returns the same number every draw; that is the failure
    # mode a mere "file exists" check would sail past.
    assert _draws.std() > 0, f"{_key}: zero variance — the sampler did not move"
    print(f"  {_key:22} mean={_draws.mean():.4g}  sd={_draws.std():.4g}  n={_draws.size}")

print(f"\n[NB03 self-check OK] 4 surrogates coupled, {len(_vars)} variables sampled")